# Chapter 7 — The Physical Design Frontier: Challenges & Bottlenecks

**Multi-Agent Analog EDA — PhD-Level Monograph (Notebook Form)**

---

Multi-agent analog EDA systems must close loops through **layout**, **extraction**, **simulation**, and **sign-off**. Unlike digital flows where Boolean correctness dominates, analog performance is a **continuous functional** of geometry, process, and environment.
This chapter formalizes the **hardware-specific constraints** that make physical design the dominant bottleneck for agentic loops: slow evaluators, layout-dependent effects (LDE), data scarcity, hard physics limits, toolchain heterogeneity, and verification fragility.

### Learning objectives

1. **Model** the *feedback-loop crisis* as a constrained time budget over expensive SPICE corners.
2. **Quantify** WPE/STI/matching and parasitic back-annotation as perturbations on small-signal and transient metrics.
3. **Analyze** the *data wall* for circuit foundation models (CFMs) and synthetic/transfer remedies.
4. **Relate** Joule heating, EM, RF crosstalk, and photonic nonlinearity to **physics-grounded** agent reasoning.
5. **Map** open layout stacks (ALIGN, OpenROAD) and **constraint-driven** analog layout to agent orchestration.
6. **Design** agent policies for DRC/LVS proofreading and iterative placement/routing under LDE.

### Notation

- Design parameters $\theta$ (sizes, biases, spacing), layout realization $L(\theta)$, extracted netlist $\mathcal{E}(L)$.
- True performance $y^\star(\theta) = f_{\text{SPICE}}(\mathcal{E}(L(\theta)), \text{corners})$.
- Surrogate $\hat{y}_\phi$ with inference cost $c_\phi \ll c_{\text{SPICE}}$.
- Agent policy $\pi$ selects actions $a_t$ (edit layout, change constraints, request simulation) under budget $B$.

---


## 7.1 The feedback-loop crisis

**Observation:** *generation* (netlist edits, macro placement, constraint scripts) is comparatively fast, while **evaluation** via full-chip / block-level SPICE across **PVT corners** and **LDE-aware extraction** is slow. A single heavy corner can consume **hours** of CPU when including post-layout extraction, aging, and statistical corners.

### 7.1.1 Time-budget model

Let $K$ be the number of SPICE jobs in one "outer" agent iteration (corners × tests). Let $T_k$ be wall time for job $k$, and $c_{\text{gen}}$ the amortized generation time. Over $N$ outer iterations with budget $B$:

$$
\sum_{n=1}^{N} \Big( c_{\text{gen}} + \sum_{k \in \mathcal{S}_n} T_k \Big) \le B,
$$

where $\mathcal{S}_n$ is the subset of simulations chosen at iteration $n$. If agents use *full* evaluation ($\mathcal{S}_n = \{1,\ldots,K\}$) and $T_k \sim \bar{T}$, then $N \lesssim B / (c_{\text{gen}} + K\bar{T})$. **Acceleration** replaces some jobs with cheap proxies:

$$
\hat{T}_k = \mathbb{1}[\text{full}_k]\, T_k + \mathbb{1}[\text{surrogate}_k]\, \tau_k, \quad \tau_k \ll T_k.
$$

**Risk:** surrogates introduce bias $\Delta_k = \mathbb{E}[\hat{y}_\phi - y^\star]$ and variance; the agent must optimize under **accuracy–cost** tradeoffs.

### 7.1.2 Incremental simulation

When layout edits are local, **incremental extraction** and **hierarchical simulation** reuse unchanged partitions. Conceptually,

$$
T_{\text{incr}} \approx \alpha \, T_{\text{full}}, \quad \alpha \in (0,1),
$$

with $\alpha$ depending on fraction of nets/cells touched and matrix reuse in the SPICE Jacobian.

---


In [ ]:
import sys; sys.path.insert(0, '..')
from style_utils import (setup_3b1b_style, glow_line, glow_fill, styled_box,
                         styled_arrow, finish_plot, plotly_3b1b_layout,
                         BACKGROUND, SURFACE, TEXT, TEXT_DIM, GRID,
                         BLUE, TEAL, GREEN, YELLOW, GOLD, RED,
                         ROSE, PURPLE, CYAN, ORANGE, PALETTE)
setup_3b1b_style()

# Global imports, dark theme (matplotlib #0d1117, plotly_dark)
from __future__ import annotations

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from scipy import stats

import plotly.graph_objects as go
import plotly.io as pio

RNG = np.random.default_rng(7)

DARK_BG = "#0d1117"
ACCENT = "#58a6ff"
GREEN = "#3fb950"
ORANGE = "#d29922"
RED = "#f85149"
PURPLE = "#bc8cff"

mpl.rcParams.update(
    {
        "figure.facecolor": DARK_BG,
        "axes.facecolor": DARK_BG,
        "axes.edgecolor": "#30363d",
        "axes.labelcolor": "#c9d1d9",
        "text.color": "#c9d1d9",
        "xtick.color": "#8b949e",
        "ytick.color": "#8b949e",
        "grid.color": "#21262d",
        "grid.alpha": 0.65,
        "legend.facecolor": "#161b22",
        "legend.edgecolor": "#30363d",
        "font.size": 11,
    }
)
pio.templates.default = "plotly_dark"

print("Ready: numpy/scipy/matplotlib/plotly (dark themes).")


In [ ]:
# --- 7.1.3 Visualize: outer iterations vs budget; surrogate Pareto (time vs RMSE) ---

def iterations_under_budget(B_hours: float, c_gen_min: float, K: int, T_bar_hours: float,
                            use_surrogate_frac: float, tau_hours: float) -> float:
    """Approximate feasible outer iterations (continuous relaxation)."""
    T_eff = (1 - use_surrogate_frac) * K * T_bar_hours + use_surrogate_frac * K * tau_hours
    denom = (c_gen_min / 60) + T_eff  # c_gen in minutes
    return B_hours / denom


B = 48.0  # wall-clock hours for one design sprint
K_corners = 64
T_bar = 1.5  # hours per heavy corner (order-of-magnitude plausible for extracted block + MC)
c_gen = 3.0  # minutes per agent iteration generation
tau = 0.01  # surrogate "corner" in hours (~36 s)

fracs = np.linspace(0, 0.95, 80)
iters_full = np.array([iterations_under_budget(B, c_gen, K_corners, T_bar, f, tau) for f in fracs])

# Toy RMSE of surrogate vs fraction of full SPICE retained (power-law saturation)
rmse = 0.45 * (1 - fracs) ** 0.7 + 0.02

fig, ax = plt.subplots(figsize=(9, 5), dpi=120)
ax.plot(fracs, iters_full, color=ACCENT, lw=2.2, label="Feasible outer iterations $N$")
ax.set_xlabel("Fraction of corners evaluated with surrogate")
ax.set_ylabel("$N$ under budget (approx.)")
ax.set_title("Feedback-loop crisis: surrogate use increases iterations but risks bias")
ax.grid(True)
ax2 = ax.twinx()
ax2.plot(fracs, rmse, color=ORANGE, ls="--", lw=2, label="Toy surrogate RMSE")
ax2.set_ylabel("Surrogate error (toy)")
ax2.tick_params(axis="y", colors=ORANGE)
for spine in ax2.spines.values():
    spine.set_edgecolor("#30363d")
fig.legend(loc="upper center", ncol=2, bbox_to_anchor=(0.5, -0.02))
fig.tight_layout()
plt.show()

# Plotly: Pareto-style scatter — mean eval time per iteration vs expected error
np.random.seed(7)
n = 120
proxy_load = np.random.beta(2, 5, n)
time_h = 0.05 + proxy_load * K_corners * T_bar * 0.25 + (1 - proxy_load) * K_corners * tau
err = 0.03 + 0.5 * proxy_load**1.2 + 0.02 * np.random.randn(n)

fig2 = go.Figure(
    go.Scatter(
        x=time_h,
        y=np.maximum(err, 0.01),
        mode="markers",
        marker=dict(size=7, color=proxy_load, colorscale="Viridis", showscale=True, colorbar=dict(title="Proxy load")),
        text=[f"policy {i}" for i in range(n)],
        name="policies",
    )
)
fig2.update_layout(
    title="Simulation time vs accuracy (toy Monte Carlo over agent policies)",
    xaxis_title="Mean evaluator time per outer iteration (h)",
    yaxis_title="Expected metric error (toy)",
    paper_bgcolor=DARK_BG,
    plot_bgcolor="#161b22",
)
fig2.show()


### 7.1.3 Surrogate acceleration: bias–variance–cost

A common decomposition for a surrogate $\hat{y}_\phi$ trained on $\mathcal{D}$ is

$$
\mathbb{E}\big[(\hat{y}_\phi - y^\star)^2\big] = \underbrace{\big(\mathbb{E}[\hat{y}_\phi] - y^\star\big)^2}_{\text{bias}^2} + \underbrace{\mathrm{Var}(\hat{y}_\phi)}_{\text{variance}} + \sigma_\epsilon^2,
$$

while **inference cost** $c_\phi$ may scale sublinearly in the number of devices. **Calibration** maps surrogate outputs to confidence intervals (e.g., Gaussian processes, deep ensembles), enabling **risk-aware** corner pruning: simulate fully only when upper confidence bounds violate specs.

**Incremental simulation** can be modeled as $T_{\mathrm{incr}} / T_{\mathrm{full}} \approx \alpha(f)$ with touched fraction $f$ of nets/devices:

$$
\alpha(f) = \alpha_{\min} + (1-\alpha_{\min})\, f^{\beta}, \quad f \in [0,1],
$$

a phenomenological curve capturing reuse of factorized matrices / hierarchical partitions.

---


In [ ]:
# --- 7.1.4 Incremental eval: alpha vs touched fraction; calibrated vs naive surrogate ---

def alpha_of_touch(f: np.ndarray, amin: float = 0.08, beta: float = 1.25) -> np.ndarray:
    return amin + (1.0 - amin) * np.power(np.clip(f, 0, 1), beta)


f = np.linspace(0, 1, 200)
fig, ax = plt.subplots(figsize=(8, 4.2), dpi=120)
ax.plot(f, alpha_of_touch(f), color=ACCENT, lw=2.4, label=r"$\alpha(f)$ — time ratio $T_{\mathrm{incr}}/T_{\mathrm{full}}$")
ax.set_xlabel("touched fraction $f$ of netlist/parasitics")
ax.set_ylabel(r"$\alpha$")
ax.set_title("Incremental simulation: more locality ⇒ stronger reuse (toy)")
ax.grid(True)
ax.legend()
plt.show()

# Calibrated (narrower) vs naive uncertainty vs training points
m = np.linspace(50, 5000, 120)
sigma_naive = 0.55 / np.sqrt(m) + 0.04
sigma_cal = 0.35 / np.sqrt(m) + 0.025

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=m, y=sigma_naive, mode="lines", name="naive i.i.d. error scale", line=dict(color=ORANGE)))
fig2.add_trace(go.Scatter(x=m, y=sigma_cal, mode="lines", name="calibrated / active sampling (toy)", line=dict(color=GREEN)))
fig2.update_layout(
    title="Surrogate uncertainty vs training points (schematic)",
    xaxis_title="labeled SPICE points",
    yaxis_title="RMSE on held-out corners (toy)",
    yaxis_type="log",
    xaxis_type="log",
)
fig2.show()


## 7.2 Layout interdependency (LDE)

Analog metrics (bandwidth, noise, offset, PSRR) are **functionals** of the extracted RC network and **stress/proximity** parameters embedded in compact models.

### 7.2.1 Well proximity effect (WPE)

Let $d_{\mathrm{nw}}$ be distance to the nearest deep N-well edge. A common *phenomenological* shift for threshold voltage:

$$
\Delta V_{\mathrm{th,WPE}} \approx \frac{\beta_{\mathrm{WPE}}}{\bigl(1 + d_{\mathrm{nw}}/L_{\mathrm{char}}\bigr)^{\gamma}},
$$

with technology-specific $(\beta_{\mathrm{WPE}}, L_{\mathrm{char}}, \gamma)$. Agents must treat $\Delta V_{\mathrm{th}}$ as a **layout decision variable** coupled to matching pairs.

### 7.2.2 STI stress

Active-to-STI distance $d_{\mathrm{STI}}$ modulates mobility and $V_{\mathrm{th}}$ via mechanical stress from oxide fill. Foundries capture this through **stress-aware** SPICE models (instance parameters from extraction).

### 7.2.3 Matching and parasitic back-annotation

For a differential pair, mismatch variance often scales as

$$
\sigma^2_{\Delta} \propto \frac{1}{WL} + \sigma^2_{\text{grad}} \, d^2,
$$

where $d$ is pair spacing and $\sigma^2_{\text{grad}}$ models **deterministic gradients** across wafer/die. **Parasitic extraction** produces $R_{ij}, C_{ij}, L_{ij}$; **back-annotation** updates the netlist so $f_{\text{SPICE}}$ reflects layout.

---


In [ ]:
# --- 7.2.4 Toy: WPE + STI shifts + parasitic pole pushing GBW ---

def vth_wpe(d_um: np.ndarray, beta_mv: float = 18.0, Lc_um: float = 0.35, gamma: float = 1.1) -> np.ndarray:
    return -beta_mv / (1.0 + d_um / Lc_um) ** gamma  # mV, nMOS-style illustrative


def vth_sti(sa_um: np.ndarray, sb_um: np.ndarray, k_sa: float = 2.5, k_sb: float = 2.0) -> np.ndarray:
    # Very coarse separable model (mV): closer STI -> more stress
    return k_sa / (sa_um + 0.05) + k_sb / (sb_um + 0.05)


d_grid = np.linspace(0.15, 3.0, 200)
fig, ax = plt.subplots(1, 2, figsize=(11, 4), dpi=120)
ax[0].plot(d_grid, vth_wpe(d_grid), color=ACCENT, lw=2)
ax[0].set_title("Illustrative WPE-like $V_{th}$ shift")
ax[0].set_xlabel("$d_{nw}$ ($\\mu$m)")
ax[0].set_ylabel("$\\Delta V_{th}$ (mV)")
ax[0].grid(True)

sa = np.linspace(0.1, 1.0, 80)
SB, SA = np.meshgrid(sa, sa)
Z = vth_sti(SA, SB)
cmap = LinearSegmentedColormap.from_list("ice", ["#010409", ACCENT])
cf = ax[1].contourf(SA, SB, Z, levels=18, cmap=cmap)
ax[1].set_title("Toy STI-distance stress surface ($\\Delta V_{th}$)")
ax[1].set_xlabel("$S_A$ ($\\mu$m)")
ax[1].set_ylabel("$S_B$ ($\\mu$m)")
fig.colorbar(cf, ax=ax[1], label="mV")
fig.tight_layout()
plt.show()

# Parasitics degrade small-signal GBW: g_m / (2*pi*C_L_eff), C_L_eff = C_L + C_wire + C_gate_next
gm = 1.2e-3  # S
CL = 120e-15
Cpar = np.linspace(0, 400e-15, 300)
gbw_mhz = (gm / (2 * np.pi * (CL + Cpar))) / 1e6

figp, axp = plt.subplots(figsize=(7.5, 4.2), dpi=120)
axp.plot(Cpar * 1e15, gbw_mhz, color=GREEN, lw=2.2)
axp.set_xlabel("Added layout parasitic capacitance (fF)")
axp.set_ylabel("GBW (MHz) — toy OTA pole")
axp.set_title("Layout parasitics monotonically erode bandwidth")
axp.grid(True)
figp.tight_layout()
plt.show()


#### 7.2.5 Matching as a spatial stochastic process

Pair a left device $p$ and right device $q$ at layout positions $\mathbf{r}_p, \mathbf{r}_q$. A *gradient* model for parameter difference is

$$
\Delta \theta_{pq} = \underbrace{\eta_{pq}}_{\text{local mismatch}} + \underbrace{\mathbf{g}^\top (\mathbf{r}_p - \mathbf{r}_q)}_{\text{deterministic gradient}} + \text{higher order},
$$

with $\mathbb{E}[\eta_{pq}]=0$, $\mathrm{Var}(\eta_{pq}) \propto 1/(WL)$. **Common-centroid** and **cross-coupled** layouts zero first-order gradient terms at the expense of routing parasitics — the agent must optimize the **composite** objective.

#### 7.2.6 Parasitic extraction and back-annotation

Let $\mathcal{G}$ be the schematic graph. Extraction produces a weighted graph $\mathcal{G}' = (\mathcal{V}, \mathcal{E}')$ with $\mathcal{E}' = \{R_{ij}, C_{ij}, L_{ij}, \ldots\}$ from field solvers or pattern matching. **Back-annotation** is the morphism

$$
\mathcal{B}: \mathcal{G}' \rightarrow \text{SPICE netlist instance parameters},
$$

so that $f_{\text{SPICE}} \circ \mathcal{B} \circ \mathcal{E}$ is the post-layout evaluator $y^\star$.

---


In [ ]:
# --- 7.2.7 Monte Carlo: mismatch-induced offset (toy OTA) ---

def mc_offset(n: int = 8000, wl_um2: float = 2.0, grad_mv_per_um: float = 0.35, sep_um: float = 1.2):
    sigma_local_mv = 1.2 / np.sqrt(wl_um2)  # Pelgrom-style scale (toy)
    eta = RNG.normal(0, sigma_local_mv, size=n)
    grad = grad_mv_per_um * sep_um * np.sin(RNG.uniform(-np.pi / 6, np.pi / 6, size=n))
    return eta + grad


offs = mc_offset()
fig, ax = plt.subplots(figsize=(8, 4.2), dpi=120)
ax.hist(offs, bins=60, density=True, color=ACCENT, alpha=0.85, edgecolor="#30363d")
xs = np.linspace(offs.min(), offs.max(), 300)
ax.plot(xs, stats.norm.pdf(xs, loc=np.mean(offs), scale=np.std(offs)), color=GREEN, lw=2, label="Gaussian fit")
ax.set_title("Toy distribution: local mismatch + deterministic gradient jitter")
ax.set_xlabel("input-referred offset (mV)")
ax.set_ylabel("density")
ax.legend()
ax.grid(True, ls=":", alpha=0.5)
plt.show()


## 7.3 The data wall

Industrial **golden** designs, extracted netlists, and correlated PDK data are **trade secrets**. Public analog datasets are tiny compared to NLP/CV corpora, limiting supervised pretraining for **circuit foundation models (CFMs)**.

Let $\mathcal{D}_{\text{real}}$ be available labeled pairs (design, metrics). A CFM minimizes

$$
\mathcal{L}(\phi) = \mathbb{E}_{(x,y)\sim \mathcal{D}_{\text{real}}} \ell\big(\hat{y}_\phi(x), y\big) + \lambda \, \mathbb{E}_{(\tilde{x},\tilde{y})\sim \mathcal{D}_{\text{syn}}} \ell\big(\hat{y}_\phi(\tilde{x}), \tilde{y}\big),
$$

where $\mathcal{D}_{\text{syn}}$ is **synthetic** data from SPICE sweeps / generative models. **Transfer** from related domains (digital timing graphs, PCB, EM sim) supplies inductive bias but shifts the feature distribution.

### 7.3.1 Order-of-magnitude scale

- Large language corpora: $10^{12}$+ tokens.
- Public structured circuit corpora: often $10^3$–$10^5$ **examples** (orders below NLP).
- Each "example" may require **expensive labels** (full extraction + corners).

The **sample complexity** for regression to error $\epsilon$ with Lipschitz surrogate class $\mathcal{H}$ scales roughly as $n \gtrsim C/\epsilon^d$ (curse of dimensionality in netlist+layout features).

---


In [ ]:
# --- 7.3.2 Quantify data scarcity vs model capacity (toy scaling) ---

def generalization_gap(n: float, d: int = 24, noise: float = 0.08) -> float:
    """Phenomenological gap ~ a / n^{1/d} + noise floor."""
    return 0.85 / (n ** (1.0 / max(d, 1))) + noise


n_vals = np.logspace(1.8, 6, 120)
dims = [8, 16, 32]
fig, ax = plt.subplots(figsize=(8.5, 4.8), dpi=120)
for d in dims:
    ax.loglog(n_vals, generalization_gap(n_vals, d=d), lw=2, label=f"feature dim $d={d}$")
ax.set_xlabel("labeled designs $n$ (log scale)")
ax.set_ylabel("toy generalization error")
ax.set_title("Data wall: error floor vs dataset size (illustrative scaling)")
ax.legend()
ax.grid(True, which="both", ls=":", alpha=0.5)
plt.show()

# Synthetic + transfer: reduce effective dimension via pretraining
pretrain = np.linspace(0, 1, 50)
err = 0.35 * (1 - pretrain) ** 1.6 + 0.055

fig2 = go.Figure()
fig2.add_trace(
    go.Scatter(
        x=pretrain,
        y=err,
        mode="lines",
        line=dict(color=ACCENT, width=3),
        name="toy error",
    )
)
fig2.update_layout(
    title="Transfer / synthetic pretraining lowers error (schematic)",
    xaxis_title="Fraction of capacity aligned to related pretraining",
    yaxis_title="Downstream analog prediction error (toy)",
)
fig2.show()


## 7.4 Physical limits

Agents cannot negotiate with **physics**. The following are *exemplars* MAS must treat as **hard constraints** or **explicit uncertainty**, not suggestions.

### 7.4.1 Joule heating and thermal management

Local power density $q(\mathbf{r}) = \mathbf{J}(\mathbf{r}) \cdot \mathbf{E}(\mathbf{r})$ sources the heat equation

$$
\rho c_p \, \frac{\partial T}{\partial t} = \nabla \cdot \big( \kappa(T) \nabla T \big) + q(\mathbf{r}, t).
$$

Mobility and threshold voltage shift with $T$, so **electro-thermal co-design** is mandatory at high current density (e.g., drivers, LDO pass devices).

### 7.4.2 RF crosstalk (analog/digital coexistence)

A minimal *linear* model for substrate coupling from a digital aggressor voltage $v_{\mathrm{dig}}(t)$ to a sensitive analog node $v_{\mathrm{ana}}(t)$ is

$$
V_{\mathrm{ana}}(j\omega) = H_{\mathrm{sub}}(j\omega)\, V_{\mathrm{dig}}(j\omega) + N(j\omega),
$$

where $H_{\mathrm{sub}}$ aggregates **capacitive** and **resistive** paths through the epi/bulk and package. Clock harmonics create **structured** interference; layout mitigations include **guard rings**, **deep N-well isolation**, **stitching**, and **current-return engineering**.

### 7.4.3 Photonic nonlinearity at advanced nodes (co-packaged optics)

In integrated photonics, refractive index depends on field intensity (Kerr effect), $n = n_0 + n_2 I$. For propagation length $L$, a common *phase* nonlinearity scales as

$$
\Delta \phi_{\mathrm{NL}} \propto \frac{2\pi}{\lambda} n_2 I \, L,
$$

producing **cross-phase modulation** and **four-wave mixing** in dense WDM. Electronic–photonic co-design agents must budget **OSNR / BER** penalties from these interactions, not only bandwidth.

### 7.4.4 Electromigration (EM)

Black's equation (empirical) captures the sharp lifetime sensitivity to current density $J$ and temperature $T$:

$$
\mathrm{MTTF} \propto \frac{1}{J^{n}} \exp\!\Big(\frac{E_a}{k_B T}\Big).
$$

Layout synthesis must allocate **via stacks** and **metal widths** so EM margins hold at **worst-case** activity.

### 7.4.5 Physics-grounded reasoning

**Physics-grounded** agents expose *invariants* as tool-checkable predicates: peak $|J|$, max $T$, crosstalk budget, Kerr phase cap, etc. Optimization is then *constrained* feasibility, not unconstrained curve fitting.

---


In [ ]:
# --- 7.4.1 Steady-state thermal map (2D finite-difference) + current density ---

def solve_laplace_heat(n: int = 81, q_center: float = 1.0):
    """Poisson: -∇²T = q on unit square, T=0 on boundaries; uniform k=1."""
    h = 1.0 / (n - 1)
    N = n * n
    A = np.zeros((N, N))
    for j in range(n):
        for i in range(n):
            idx = j * n + i
            if i in (0, n - 1) or j in (0, n - 1):
                A[idx, idx] = 1.0
                continue
            A[idx, idx] = -4.0
            A[idx, idx - 1] = 1.0
            A[idx, idx + 1] = 1.0
            A[idx, idx - n] = 1.0
            A[idx, idx + n] = 1.0
    xs = np.linspace(0, 1, n)
    X, Y = np.meshgrid(xs, xs)
    q = q_center * np.exp(-((X - 0.55) ** 2 + (Y - 0.45) ** 2) / (2 * 0.07**2))
    b = np.zeros(N)
    for j in range(n):
        for i in range(n):
            idx = j * n + i
            if i in (0, n - 1) or j in (0, n - 1):
                b[idx] = 0.0
            else:
                b[idx] = -(h**2) * q[j, i]
    T_vec = np.linalg.solve(A, b)
    T = T_vec.reshape(n, n)
    return X, Y, q, T


X, Y, q, T = solve_laplace_heat(n=81, q_center=80.0)

fig, ax = plt.subplots(1, 3, figsize=(12.5, 3.8), dpi=120)
im0 = ax[0].imshow(q, origin="lower", extent=[0, 1, 0, 1], cmap="magma")
ax[0].set_title("Power density $q(x,y)$ (toy)")
fig.colorbar(im0, ax=ax[0], fraction=0.046, pad=0.04)
im1 = ax[1].imshow(T, origin="lower", extent=[0, 1, 0, 1], cmap="inferno")
ax[1].set_title("Steady-state $T$ (FD solve, $T=0$ on boundary)")
fig.colorbar(im1, ax=ax[1], fraction=0.046, pad=0.04)

# Current density crowding (toy analytic mask)
W, H = 2.0, 1.0
nx, ny = 200, 120
x = np.linspace(0, W, nx)
y = np.linspace(0, H, ny)
XG, YG = np.meshgrid(x, y)
phi = np.hypot((XG - 0.85 * W) / W, (YG - 0.75 * H) / H)
Jmag = 1.0 + 2.2 * np.exp(-phi**2 / (2 * 0.18**2))
im2 = ax[2].contourf(XG, YG, Jmag, levels=28, cmap="cividis")
ax[2].set_title("Toy $|J|$ crowding (EM hotspot proxy)")
ax[2].set_xlabel("x")
ax[2].set_ylabel("y")
fig.colorbar(im2, ax=ax[2])
for a in ax:
    a.set_facecolor(DARK_BG)
fig.tight_layout()
plt.show()

# Plotly surface for temperature
fig2 = go.Figure(
    data=[
        go.Surface(
            z=T,
            x=np.linspace(0, 1, T.shape[1]),
            y=np.linspace(0, 1, T.shape[0]),
            colorscale="Hot",
            showscale=True,
        )
    ]
)
fig2.update_layout(
    title="Thermal map (surface view)",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="T"),
    margin=dict(l=0, r=0, t=50, b=0),
)
fig2.show()


In [ ]:
# --- 7.4.6 RF substrate crosstalk + photonic Kerr phase (toy plots) ---

omega = np.logspace(7, 11, 600)  # rad/s
# Toy: two-pole substrate coupling with roll-off
H = 1e-3 / ((1j * omega / 5e8 + 1) * (1j * omega / 3e9 + 1))
fig, ax = plt.subplots(figsize=(8.5, 4.3), dpi=120)
ax.loglog(omega / (2 * np.pi), np.abs(H), color=PURPLE, lw=2.2)
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel(r"$|H_{\mathrm{sub}}(j\omega)|$ (toy)")
ax.set_title("Illustrative substrate transfer: digital noise couples more at mid-band")
ax.grid(True, which="both", ls=":", alpha=0.55)
plt.show()

# Kerr phase vs intensity (plotly)
lam_nm = 1550.0
n2_eff = 2.5e-14  # m^2/W toy effective
L_m = 0.005
I_w_per_m2 = np.linspace(1e4, 2e6, 120)
delta_phi = (2 * np.pi / (lam_nm * 1e-9)) * n2_eff * I_w_per_m2 * L_m

fig2 = go.Figure(
    go.Scatter(
        x=I_w_per_m2 / 1e6,
        y=delta_phi,
        mode="lines",
        line=dict(color=ACCENT, width=3),
        name="Δφ",
    )
)
fig2.update_layout(
    title="Photonic Kerr-like phase shift vs intensity (illustrative)",
    xaxis_title="intensity (MW/m², toy scale)",
    yaxis_title="nonlinear phase (rad)",
)
fig2.show()


## 7.5 Layout automation tools

### 7.5.1 ALIGN (Analog Layout Intelligently Generated Automatically)

**ALIGN** targets **automatic analog layout** from SPICE-like constraints using hierarchical assembly and constraint propagation — valuable as an **open** generator for agent loops (generate → DRC/LVS → extract → simulate).

### 7.5.2 OpenROAD for digital portions

Mixed-signal SoCs pair analog macros with digital. **OpenROAD** provides open-place-route for the digital **sea-of-cells** region; agents must **partition** responsibilities: analog macro generators vs digital PnR.

### 7.5.3 Constraint-driven analog layout

Constraints include symmetry, proximity, matching, IR-drop limits, and shielding. Formally, layout is

$$
L^\star \in \arg\min_{L \in \mathcal{L}} \; \mathcal{C}(L) \quad \text{s.t.} \quad g_i(L) \le 0,\; h_j(L) = 0,
$$

where $\mathcal{C}$ is a proxy cost (wirelength, area) and $g_i$ encode **electrical** budgets from sensitivity analysis.

### 7.5.4 DRC/LVS verification loops

**DRC** checks geometric rules; **LVS** checks connectivity vs schematic. Agentic flows treat tool reports as **typed observations** $o_t$ and branch on error classes (short vs spacing vs device mismatch).

---


In [ ]:
# --- 7.5.5 Flow diagram: ALIGN + OpenROAD + sign-off (Plotly Sankey) ---

fig = go.Figure(
    data=[
        go.Sankey(
            arrangement="snap",
            node=dict(
                pad=18,
                thickness=18,
                line=dict(color="#30363d", width=1),
                label=[
                    "Netlist + constraints",
                    "ALIGN (analog)",
                    "OpenROAD (digital)",
                    "Merged GDS",
                    "DRC/LVS",
                    "RCX + back-annot.",
                    "Corners SPICE",
                ],
                color=[PURPLE, ACCENT, GREEN, ORANGE, RED, "#79c0ff", "#ffa657"],
            ),
            link=dict(
                source=[0, 0, 3, 4, 5, 5],
                target=[1, 2, 4, 5, 6, 6],
                value=[35, 45, 80, 70, 40, 30],
            ),
        )
    ]
)
fig.update_layout(title="Toy mixed-signal tool flow (Sankey thickness ∝ notional effort)")
fig.show()


## 7.6 Agent strategies for physical design

### 7.6.1 Handling layout-dependent effects

- **Parameterize** LDE-relevant distances in the action space $(S_A, S_B, d_{nw}, \ldots)$ and couple them to **pre-PEX** sensitivity estimates.
- **Schedule** "LDE-heavy" full extraction only when surrogate uncertainty exceeds a threshold (active learning).

### 7.6.2 Proofreading DRC/LVS

Treat verification as **structured parsing**: cluster violations by layer/rule, rank by **electrical criticality** (e.g., spacing on high-swing nets), and generate **minimal** polygon edits. Formally, choose edit $\delta L$ minimizing $\|\delta L\|$ s.t. DRC$(L+\delta L)=\checkmark$.

### 7.6.3 Iterative placement/routing optimization

Let $x_t$ be placer state (positions, route topology). A Gauss–Seidel style mental model:

$$
x_{t+1} = x_t - \eta_t \, \big( \nabla_x \mathcal{C}(x_t) + \sum_i \lambda_i \nabla_x g_i^+(x_t) \big),
$$

where $g_i^+$ are penalties for violated constraints. Agents alternate **continuous** updates with **discrete** rip-up/reroute decisions.

---


In [ ]:
# --- 7.6.4 Simulated iterative placement under LDE + DRC penalties ---

def cost_surface(xy: np.ndarray) -> float:
    x, y = xy
    # Nominal wirelength + overlap penalty + WPE proxy (distance to dummy well at 0.6,0.4)
    wl = (x - 0.25) ** 2 + (y - 0.25) ** 2
    wpe = 1.0 / (1.0 + np.hypot(x - 0.6, y - 0.4) / 0.2)  # larger when close to well edge proxy
    overlap = max(0.0, 0.18 - np.hypot(x - y, y - 0.5 * x)) * 5.0
    return wl + 0.35 * wpe + overlap


def proj_box(xy: np.ndarray, lo: float = 0.05, hi: float = 0.95) -> np.ndarray:
    return np.clip(xy, lo, hi)


path = []
xy = np.array([0.88, 0.12])
eta = 0.15
rng = np.random.default_rng(3)
for t in range(60):
    path.append(xy.copy())
    eps = 1e-4
    ex = (cost_surface(xy + np.array([eps, 0])) - cost_surface(xy - np.array([eps, 0]))) / (2 * eps)
    ey = (cost_surface(xy + np.array([0, eps])) - cost_surface(xy - np.array([0, eps]))) / (2 * eps)
    g = np.array([ex, ey]) + 0.03 * rng.standard_normal(2)
    xy = proj_box(xy - eta * g)
    eta *= 0.97

path = np.array(path)
XG = np.linspace(0, 1, 160)
YG = np.linspace(0, 1, 160)
Z = np.array([[cost_surface(np.array([xv, yv])) for xv in XG] for yv in YG])

fig, ax = plt.subplots(figsize=(7.2, 6), dpi=120)
cf = ax.contourf(XG, YG, Z, levels=28, cmap="viridis", alpha=0.95)
ax.plot(path[:, 0], path[:, 1], color=RED, lw=2, marker="o", ms=3, label="agent descent path")
ax.scatter([path[0, 0]], [path[0, 1]], color=ACCENT, s=80, label="start")
ax.scatter([path[-1, 0]], [path[-1, 1]], color=GREEN, s=90, label="end")
ax.set_title("Toy constrained placement: smooth cost + LDE-like term + noise")
ax.set_xlabel("macro x")
ax.set_ylabel("macro y")
ax.legend(loc="upper right")
fig.colorbar(cf, ax=ax, label="proxy cost")
plt.show()

# Convergence of cost (proof of iterative improvement)
costs = [cost_surface(p) for p in path]
fig2 = go.Figure(go.Scatter(y=costs, mode="lines+markers", line=dict(color=ACCENT)))
fig2.update_layout(
    title="Iteration index vs proxy cost (toy)",
    xaxis_title="t",
    yaxis_title="cost",
)
fig2.show()


## 7.7 Synthesis: what MAS must internalize

| Bottleneck | Core tension | Agentic lever |
|------------|--------------|---------------|
| Slow SPICE | Exploration vs wall-clock | Surrogates, corner pruning, incremental PEX |
| LDE | Geometry ↔ parameters | Constraint parameters in action space; LDE-aware datasets |
| Data wall | CFM sample complexity | Synthetic SPICE farms; transfer from related graphs |
| Physics | Hard feasibility | EM/thermal/crosstalk budgets as guardrails |
| Tools | Heterogeneous APIs | Typed tools: ALIGN/OpenROAD/RCX as black boxes with schemas |
| Verification | Noisy discrete failures | Proofread-by-clusters; minimal polygon diffs |

**Closing thought:** physical design is not a post-processing step — it is the **realization** of the continuous mathematics agents optimize. Treating layout as first-class state is the difference between *chatting about circuits* and *shipping silicon*.

---


In [ ]:
# --- 7.8 Optional: Black EM scaling curve (log-log) ---

J = np.logspace(4, 7, 200)  # A/m^2 scale (toy)
Ea, kB = 0.9 * 1.602176634e-19, 1.380649e-23  # J
T = 330.0
n_black = 1.8
mttf = (1e6 / J**n_black) * np.exp(Ea / (kB * T))  # arbitrary prefactor for visualization

fig, ax = plt.subplots(figsize=(7.5, 4.5), dpi=120)
ax.loglog(J, mttf, color=PURPLE, lw=2.3)
ax.set_xlabel("current density $J$ (toy units)")
ax.set_ylabel("relative MTTF (Black-style)")
ax.set_title("Electromigration: sharp lifetime penalty vs $J$")
ax.grid(True, which="both", ls=":", alpha=0.55)
plt.show()
